
# DinoV2 Dogs-vs-Cats Hypothesis: Linear Probe vs Baseline vs Augmented Baseline (Parallel 2xT4)

This notebook is a producer workflow with mixed epoch budget and no checkpoint-recovery fallback.

Training sequence:
1. `linear_probe_e2` (freeze backbone, train classifier head, 2 epochs)
2. `baseline_small_e1` (baseline small config, 1 epoch)
3. `augmented_small_e1` (baseline small + stronger train augmentation, 1 epoch)

Comparison focus:
- Main hypothesis comparison is `baseline_small_e1` vs `augmented_small_e1` at the same 1-epoch budget.
- `linear_probe_e2` is a reference run to contextualize performance under a head-only setup.

Motivation for augmentation hypothesis:
- Stronger augmentation can improve early generalization by forcing invariance to crop/color/noise changes.
- It can also improve short-run stability by reducing sensitivity to spurious batch patterns.

Flow: bootstrap -> build runtime configs -> preprocess -> optional sanity -> train all -> compare all -> baseline-vs-aug analysis -> plots -> export trained runs -> hypothesis verdicts.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128')

REPO_URL = 'https://github.com/mruniverse8/kaggle-experiments-.git'
REPO_DIR = Path('/kaggle/working/kaggle-experiments-')
BRANCH = 'dogs_vs_cats_v2'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(['git', 'fetch', '--all'], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

current_branch = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).decode().strip()
current_commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip()
print('Git branch:', current_branch)
print('Git commit:', current_commit)
print('Repo ready at:', REPO_DIR)


In [ ]:

import os
import json
import copy
from pathlib import Path

PATHS_CFG = os.environ.get('PATHS_CFG', 'dogs_vs_cats/configs/paths_kaggle.json')
CFG_LINEAR = os.environ.get('CFG_LINEAR', 'dogs_vs_cats/configs/experiments/dinov2_vitb14_linear_probe.json')
CFG_BASELINE = os.environ.get('CFG_BASELINE', 'dogs_vs_cats/configs/experiments/dinov2_vitb14_parallel_t4x2_small.json')
RUN_SANITY = os.environ.get('RUN_SANITY', '0') == '1'

PATHS = json.loads(Path(PATHS_CFG).read_text())
linear_cfg_base = json.loads(Path(CFG_LINEAR).read_text())
baseline_cfg_base = json.loads(Path(CFG_BASELINE).read_text())

runtime_cfg_dir = REPO_DIR / 'dogs_vs_cats/configs/experiments/runtime'
runtime_cfg_dir.mkdir(parents=True, exist_ok=True)

def write_runtime_cfg(name: str, payload: dict) -> str:
    path = runtime_cfg_dir / f'{name}.json'
    path.write_text(json.dumps(payload, indent=2))
    return str(path.relative_to(REPO_DIR))

# linear probe: 2 epochs
linear_e2 = copy.deepcopy(linear_cfg_base)
linear_e2['experiment_name'] = f"{linear_cfg_base['experiment_name']}_e2_v51"
linear_e2.setdefault('training', {})['epochs'] = 2
linear_e2['training']['early_stopping_patience'] = max(0, int(linear_e2['training'].get('early_stopping_patience', 0)))
cfg_linear_e2_path = write_runtime_cfg('dinov2_vitb14_linear_probe_e2_v51', linear_e2)

# baseline: 1 epoch
baseline_e1 = copy.deepcopy(baseline_cfg_base)
baseline_e1['experiment_name'] = f"{baseline_cfg_base['experiment_name']}_e1_v51"
baseline_e1.setdefault('training', {})['epochs'] = 1
baseline_e1['training']['early_stopping_patience'] = 0
cfg_baseline_e1_path = write_runtime_cfg('dinov2_vitb14_parallel_t4x2_small_e1_v51', baseline_e1)

# augmented baseline: 1 epoch with stronger train augmentation
augmented_e1 = copy.deepcopy(baseline_e1)
augmented_e1['experiment_name'] = f"{baseline_cfg_base['experiment_name']}_augmented_e1_v51"
aug_train = augmented_e1.setdefault('augmentation', {}).setdefault('train', {})
aug_train['random_resized_crop_scale'] = [0.5, 1.0]
aug_train['horizontal_flip_p'] = 0.5
aug_train['color_jitter'] = [0.25, 0.25, 0.25, 0.08]
aug_train['random_erasing_p'] = 0.2
cfg_augmented_e1_path = write_runtime_cfg('dinov2_vitb14_parallel_t4x2_small_augmented_e1_v51', augmented_e1)

EXPERIMENTS = [
    {'profile': 'linear_probe_e2', 'cfg_path': cfg_linear_e2_path},
    {'profile': 'baseline_small_e1', 'cfg_path': cfg_baseline_e1_path},
    {'profile': 'augmented_small_e1', 'cfg_path': cfg_augmented_e1_path},
]

print('Using paths config:', PATHS_CFG)
print('Run sanity checks:', RUN_SANITY)
print('Runtime config dir:', runtime_cfg_dir)
print('Experiments (training order):')
for exp in EXPERIMENTS:
    payload = json.loads((REPO_DIR / exp['cfg_path']).read_text())
    exp['experiment_name'] = payload['experiment_name']
    exp['config_payload'] = payload
    print('-', exp['profile'], '->', exp['cfg_path'], '| experiment_name=', payload['experiment_name'], '| epochs=', payload.get('training', {}).get('epochs'))

for key in ['train_dir', 'eval_dir', 'test_dir', 'train_zip', 'test_zip', 'sample_submission_csv']:
    value = PATHS.get(key, '')
    if not value:
        print(f"{key}: <empty>")
        continue
    exists = Path(value).exists()
    print(f"{key}: {value} | exists={exists}")


In [ ]:
import sys
import subprocess
from pathlib import Path

manifest_dir = Path(PATHS['manifests_dir'])
required = [
    manifest_dir / 'train_manifest.csv',
    manifest_dir / 'val_manifest.csv',
    manifest_dir / 'test_manifest.csv',
]

if all(path.exists() for path in required):
    print('Preprocess skipped: manifests already exist')
else:
    # Use baseline config for split/preprocess defaults.
    subprocess.run([
        sys.executable,
        'dogs_vs_cats/src/preprocess_competition_data.py',
        '--paths-config', PATHS_CFG,
        '--experiment-config', CFG_BASELINE,
    ], check=True)


In [ ]:
import sys
import subprocess

if RUN_SANITY:
    for exp in EXPERIMENTS:
        print('Running sanity check for:', exp['experiment_name'])
        subprocess.run([
            sys.executable,
            'dogs_vs_cats/src/sanity_check_random_init.py',
            '--paths-config', PATHS_CFG,
            '--experiment-config', exp['cfg_path'],
        ], check=True)
else:
    print('Sanity checks skipped (set RUN_SANITY=1 to enable).')


In [ ]:

import sys
import json
import time
import subprocess
from datetime import datetime
from pathlib import Path

HYPOTHESIS_CONTEXT = {
    'linear_probe_e2': 'Reference run: head-only adaptation at 2 epochs.',
    'baseline_small_e1': 'Control run: 1-epoch baseline without stronger augmentation.',
    'augmented_small_e1': 'Test run: 1-epoch stronger augmentation for generalization/stability.',
}

RUNS = []
total_runs = len(EXPERIMENTS)
for run_idx, exp in enumerate(EXPERIMENTS, start=1):
    run_record = {
        'profile': exp['profile'],
        'cfg_path': exp['cfg_path'],
        'experiment_name': exp['experiment_name'],
        'status': 'failed',
        'error_message': '',
    }

    # Explicit start marker so logs clearly show when each model begins.
    start_ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    run_t0 = time.time()
    print('\n' + '=' * 90)
    print(f"[{start_ts}] START {run_idx}/{total_runs} | profile={exp['profile']} | experiment={exp['experiment_name']}")
    print('Hypothesis context:', HYPOTHESIS_CONTEXT.get(exp['profile'], 'n/a'))

    try:
        subprocess.run([
            sys.executable,
            'dogs_vs_cats/src/dinov2_pipeline.py',
            '--mode', 'train',
            '--paths-config', PATHS_CFG,
            '--experiment-config', exp['cfg_path'],
        ], check=True)

        report_path = Path(PATHS['reports_dir']) / f"{exp['experiment_name']}_training_summary.json"
        if not report_path.exists():
            raise FileNotFoundError(f'Training summary missing: {report_path}')
        summary = json.loads(report_path.read_text())

        run_record.update({
            'status': 'trained',
            'summary': summary,
            'report_path': str(report_path),
            'active_experiment_name': exp['experiment_name'],
        })
        print('Final val metrics:', summary.get('final_val_metrics', {}))
    except Exception as exc:
        run_record['status'] = 'failed'
        run_record['error_message'] = str(exc)
        print('[train] failed:', exc)

    elapsed = time.time() - run_t0
    end_ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{end_ts}] END   {run_idx}/{total_runs} | profile={exp['profile']} | status={run_record['status']} | elapsed_sec={elapsed:.1f}")

    RUNS.append(run_record)

SUCCESS_RUNS = [run for run in RUNS if run.get('status') == 'trained' and 'summary' in run]
FAILED_RUNS = [run for run in RUNS if run.get('status') == 'failed']

print('\nRUN STATUSES:')
for run in RUNS:
    print('-', run['profile'], '| status=', run.get('status'), '| active_name=', run.get('active_experiment_name', run['experiment_name']))
    if run.get('error_message'):
        print('  error:', run['error_message'])


In [ ]:

import pandas as pd

rows = []
for run in SUCCESS_RUNS:
    metrics = run['summary'].get('final_val_metrics', {})
    rows.append({
        'profile': run['profile'],
        'status': run['status'],
        'experiment_name': run.get('active_experiment_name', run['experiment_name']),
        'val_auc': metrics.get('val_auc'),
        'val_logloss': metrics.get('val_logloss'),
        'val_accuracy': metrics.get('val_accuracy'),
        'val_loss': metrics.get('val_loss'),
        'best_epoch': run['summary'].get('best_epoch'),
        'best_monitor': run['summary'].get('best_monitor'),
        'monitor': run['summary'].get('monitor'),
    })

if not rows:
    compare_df = pd.DataFrame(columns=['profile', 'status', 'experiment_name', 'val_auc', 'val_logloss', 'val_accuracy', 'val_loss', 'best_epoch', 'best_monitor', 'monitor'])
    print('No successful runs available for comparison.')
    display(compare_df)
else:
    compare_df = pd.DataFrame(rows)
    print('All successful run comparison:')
    display(compare_df.sort_values(by=['val_auc', 'val_accuracy'], ascending=[False, False]).reset_index(drop=True))


In [ ]:

import numpy as np
import pandas as pd

BASELINE_PROFILE = 'baseline_small_e1'
AUG_PROFILE = 'augmented_small_e1'

baseline_aug_compare = {
    'available': False,
    'baseline_profile': BASELINE_PROFILE,
    'augmented_profile': AUG_PROFILE,
}

if 'compare_df' not in globals() or compare_df.empty:
    print('No comparison dataframe available yet.')
else:
    base_row = compare_df.loc[compare_df['profile'] == BASELINE_PROFILE]
    aug_row = compare_df.loc[compare_df['profile'] == AUG_PROFILE]

    if len(base_row) == 1 and len(aug_row) == 1:
        base_row = base_row.iloc[0]
        aug_row = aug_row.iloc[0]

        deltas = {
            'aug_minus_base_val_auc': float(aug_row['val_auc'] - base_row['val_auc']) if pd.notna(aug_row['val_auc']) and pd.notna(base_row['val_auc']) else np.nan,
            'aug_minus_base_val_logloss': float(aug_row['val_logloss'] - base_row['val_logloss']) if pd.notna(aug_row['val_logloss']) and pd.notna(base_row['val_logloss']) else np.nan,
            'aug_minus_base_val_accuracy': float(aug_row['val_accuracy'] - base_row['val_accuracy']) if pd.notna(aug_row['val_accuracy']) and pd.notna(base_row['val_accuracy']) else np.nan,
        }

        baseline_aug_compare.update({
            'available': True,
            'baseline_experiment_name': base_row['experiment_name'],
            'augmented_experiment_name': aug_row['experiment_name'],
            **deltas,
        })

        print('Baseline vs Augmented deltas (aug - base):', deltas)
    else:
        print('Baseline and/or augmented rows not uniquely available in successful runs.')



## Hypothesis 1 (Run Right After Baseline vs Augmented Metrics)
At 1 epoch, stronger augmentation improves early generalization vs baseline.

Motivation: under a short training budget, stronger data variation can encourage invariant feature learning and improve early validation AUC/logloss.


In [ ]:

import numpy as np

HYPOTHESIS_1_VERDICT = 'INSUFFICIENT DATA'

if 'baseline_aug_compare' not in globals() or not baseline_aug_compare.get('available', False):
    print('Hypothesis 1: INSUFFICIENT DATA (baseline-vs-aug metrics not available).')
else:
    d_auc = baseline_aug_compare.get('aug_minus_base_val_auc', np.nan)
    d_log = baseline_aug_compare.get('aug_minus_base_val_logloss', np.nan)
    d_acc = baseline_aug_compare.get('aug_minus_base_val_accuracy', np.nan)
    print('Hypothesis 1 deltas (aug - base):', {
        'val_auc': d_auc,
        'val_logloss': d_log,
        'val_accuracy': d_acc,
    })
    if np.isfinite(d_auc) and np.isfinite(d_log):
        HYPOTHESIS_1_VERDICT = 'SUPPORTED' if (d_auc > 0 and d_log < 0) else 'NOT SUPPORTED'
    print('Hypothesis 1:', HYPOTHESIS_1_VERDICT)


In [ ]:

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

plot_keys = [
    'train_loss_plot',
    'val_metrics_plot',
    'grad_norm_plot',
    'val_confusion_matrix_plot',
]

if not SUCCESS_RUNS:
    print('No successful runs available to render plots.')
else:
    for run in SUCCESS_RUNS:
        print('\\nPLOTS:', run.get('active_experiment_name', run['experiment_name']))
        files = run['summary'].get('files', {})
        for key in plot_keys:
            path = files.get(key, '')
            if not path:
                continue
            p = Path(path)
            if not p.exists():
                continue
            plt.figure(figsize=(8, 4))
            plt.imshow(mpimg.imread(p))
            plt.title(f"{run['profile']} | {p.name}")
            plt.axis('off')
            plt.show()


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

if not SUCCESS_RUNS:
    print('No successful runs available for unified special comparison plot.')
else:
    run_tables = []
    for run in SUCCESS_RUNS:
        files = run['summary'].get('files', {})
        train_hist = pd.read_csv(files['train_history_csv']) if Path(files.get('train_history_csv', '')).exists() else pd.DataFrame()
        eval_hist = pd.read_csv(files['eval_history_csv']) if Path(files.get('eval_history_csv', '')).exists() else pd.DataFrame()
        metrics = run['summary'].get('final_val_metrics', {})
        run_tables.append({
            'run': run,
            'train_hist': train_hist,
            'eval_hist': eval_hist,
            'metrics': metrics,
        })

    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    ax_train, ax_auc = axes[0]
    ax_logloss, ax_summary = axes[1]

    for row in run_tables:
        name = row['run']['profile']
        train_hist = row['train_hist']
        eval_hist = row['eval_hist']
        if not train_hist.empty:
            ax_train.plot(train_hist['global_step'], train_hist['train_loss'], label=name, linewidth=2)
        if not eval_hist.empty:
            ax_auc.plot(eval_hist['global_step'], eval_hist['val_auc'], label=name, marker='o', linewidth=2)
            ax_logloss.plot(eval_hist['global_step'], eval_hist['val_logloss'], label=name, marker='o', linewidth=2)

    ax_train.set_title('Train Loss vs Global Step')
    ax_train.set_xlabel('Global step')
    ax_train.set_ylabel('train_loss')
    ax_train.grid(alpha=0.25)
    ax_train.legend()

    ax_auc.set_title('Validation AUC vs Global Step')
    ax_auc.set_xlabel('Global step')
    ax_auc.set_ylabel('val_auc')
    ax_auc.grid(alpha=0.25)
    ax_auc.legend()

    ax_logloss.set_title('Validation Logloss vs Global Step')
    ax_logloss.set_xlabel('Global step')
    ax_logloss.set_ylabel('val_logloss')
    ax_logloss.grid(alpha=0.25)
    ax_logloss.legend()

    profiles = [row['run']['profile'] for row in run_tables]
    x = np.arange(len(profiles))
    auc_vals = [row['metrics'].get('val_auc', np.nan) for row in run_tables]
    acc_vals = [row['metrics'].get('val_accuracy', np.nan) for row in run_tables]
    logloss_vals = [row['metrics'].get('val_logloss', np.nan) for row in run_tables]

    bar_w = 0.34
    bars_auc = ax_summary.bar(x - bar_w / 2, auc_vals, width=bar_w, label='val_auc')
    bars_acc = ax_summary.bar(x + bar_w / 2, acc_vals, width=bar_w, label='val_accuracy')
    ax_summary.set_ylim(0.0, 1.05)
    ax_summary.set_xticks(x, profiles)
    ax_summary.set_ylabel('AUC / Accuracy')
    ax_summary.set_title('Final Validation Metrics (All Successful Runs)')
    ax_summary.grid(alpha=0.2, axis='y')

    ax_summary_2 = ax_summary.twinx()
    ax_summary_2.plot(x, logloss_vals, color='tab:red', marker='D', linewidth=2, label='val_logloss')
    ax_summary_2.set_ylabel('Logloss (lower is better)')

    for bar in list(bars_auc) + list(bars_acc):
        height = bar.get_height()
        if np.isfinite(height):
            ax_summary.text(bar.get_x() + bar.get_width() / 2, height + 0.01, f"{height:.4f}", ha='center', va='bottom', fontsize=9)
    for xi, val in enumerate(logloss_vals):
        if np.isfinite(val):
            ax_summary_2.text(xi, val, f"{val:.4f}", color='tab:red', fontsize=9, ha='left', va='bottom')

    handles_1, labels_1 = ax_summary.get_legend_handles_labels()
    handles_2, labels_2 = ax_summary_2.get_legend_handles_labels()
    ax_summary.legend(handles_1 + handles_2, labels_1 + labels_2, loc='best')

    plt.suptitle('Linear Probe (2 Epochs) vs Baseline/Augmented (1 Epoch) - Unified Comparison', fontsize=14, y=0.98)
    plt.tight_layout()

    special_plot_path = Path(PATHS['plots_dir']) / 'v51_all_runs_special_comparison.png'
    special_plot_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(special_plot_path, dpi=160, bbox_inches='tight')
    plt.show()
    print('Saved special comparison plot to:', special_plot_path)


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

stability_rows = []
for run in SUCCESS_RUNS:
    files = run['summary'].get('files', {})
    train_hist_path = files.get('train_history_csv', '')
    train_hist = pd.read_csv(train_hist_path) if train_hist_path and Path(train_hist_path).exists() else pd.DataFrame()

    if train_hist.empty:
        row = {
            'profile': run['profile'],
            'experiment_name': run.get('active_experiment_name', run['experiment_name']),
            'train_loss_mean': np.nan,
            'train_loss_std': np.nan,
            'grad_norm_std': np.nan,
        }
    else:
        row = {
            'profile': run['profile'],
            'experiment_name': run.get('active_experiment_name', run['experiment_name']),
            'train_loss_mean': float(train_hist['train_loss'].mean()) if 'train_loss' in train_hist else np.nan,
            'train_loss_std': float(train_hist['train_loss'].std(ddof=0)) if 'train_loss' in train_hist else np.nan,
            'grad_norm_std': float(train_hist['grad_norm'].std(ddof=0)) if 'grad_norm' in train_hist else np.nan,
        }
    stability_rows.append(row)

stability_df = pd.DataFrame(stability_rows)
print('Training stability summary:')
display(stability_df)

baseline_stability = stability_df.loc[stability_df['profile'] == 'baseline_small_e1']
aug_stability = stability_df.loc[stability_df['profile'] == 'augmented_small_e1']

stability_compare = {'available': False}
if len(baseline_stability) == 1 and len(aug_stability) == 1:
    base = baseline_stability.iloc[0]
    aug = aug_stability.iloc[0]
    stability_compare = {
        'available': True,
        'aug_minus_base_train_loss_mean': float(aug['train_loss_mean'] - base['train_loss_mean']) if pd.notna(aug['train_loss_mean']) and pd.notna(base['train_loss_mean']) else np.nan,
        'aug_minus_base_train_loss_std': float(aug['train_loss_std'] - base['train_loss_std']) if pd.notna(aug['train_loss_std']) and pd.notna(base['train_loss_std']) else np.nan,
        'aug_minus_base_grad_norm_std': float(aug['grad_norm_std'] - base['grad_norm_std']) if pd.notna(aug['grad_norm_std']) and pd.notna(base['grad_norm_std']) else np.nan,
        'base_train_loss_std': float(base['train_loss_std']) if pd.notna(base['train_loss_std']) else np.nan,
        'aug_train_loss_std': float(aug['train_loss_std']) if pd.notna(aug['train_loss_std']) else np.nan,
        'base_grad_norm_std': float(base['grad_norm_std']) if pd.notna(base['grad_norm_std']) else np.nan,
        'aug_grad_norm_std': float(aug['grad_norm_std']) if pd.notna(aug['grad_norm_std']) else np.nan,
    }
    print('Stability deltas (aug - base):')
    print(stability_compare)

    # Interpretation: lower std suggests steadier optimization dynamics.
    loss_std_delta = stability_compare.get('aug_minus_base_train_loss_std', np.nan)
    grad_std_delta = stability_compare.get('aug_minus_base_grad_norm_std', np.nan)
    if np.isfinite(loss_std_delta) and np.isfinite(grad_std_delta):
        if loss_std_delta < 0 and grad_std_delta <= 0:
            print('Interpretation: augmentation appears more stable (lower/equal variability).')
        elif loss_std_delta > 0 and grad_std_delta > 0:
            print('Interpretation: augmentation appears less stable (higher variability).')
        else:
            print('Interpretation: mixed stability signal (one metric improves, one does not).')
else:
    print('Could not compute baseline-vs-aug stability deltas (missing successful rows).')



## Hypothesis 2 (Run After Stability Analysis)
At 1 epoch, stronger augmentation improves early training stability vs baseline.

Motivation: broader input diversity can smooth optimization and reduce variance in train loss and gradient norms.


In [ ]:

import numpy as np

HYPOTHESIS_2_VERDICT = 'INSUFFICIENT DATA'

if 'stability_compare' not in globals() or not stability_compare.get('available', False):
    print('Hypothesis 2: INSUFFICIENT DATA (stability comparison not available).')
else:
    base_loss_std = stability_compare.get('base_train_loss_std', np.nan)
    aug_loss_std = stability_compare.get('aug_train_loss_std', np.nan)
    base_grad_std = stability_compare.get('base_grad_norm_std', np.nan)
    aug_grad_std = stability_compare.get('aug_grad_norm_std', np.nan)
    print('Hypothesis 2 stats:', {
        'base_train_loss_std': base_loss_std,
        'aug_train_loss_std': aug_loss_std,
        'base_grad_norm_std': base_grad_std,
        'aug_grad_norm_std': aug_grad_std,
    })
    vals = [base_loss_std, aug_loss_std, base_grad_std, aug_grad_std]
    if all(np.isfinite(v) for v in vals):
        HYPOTHESIS_2_VERDICT = 'SUPPORTED' if (aug_loss_std < base_loss_std and aug_grad_std <= base_grad_std) else 'NOT SUPPORTED'
    print('Hypothesis 2:', HYPOTHESIS_2_VERDICT)


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

if not baseline_aug_compare.get('available', False):
    print('Skipping baseline-vs-aug plot: required successful runs are not available.')
else:
    base_row = compare_df.loc[compare_df['profile'] == 'baseline_small_e1'].iloc[0]
    aug_row = compare_df.loc[compare_df['profile'] == 'augmented_small_e1'].iloc[0]

    labels = ['baseline_small_e1', 'augmented_small_e1']
    auc_vals = [base_row['val_auc'], aug_row['val_auc']]
    acc_vals = [base_row['val_accuracy'], aug_row['val_accuracy']]
    log_vals = [base_row['val_logloss'], aug_row['val_logloss']]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].bar(labels, auc_vals, color=['tab:blue', 'tab:orange'])
    axes[0].set_title('Validation AUC (higher better)')
    axes[0].set_ylim(0.0, 1.05)
    axes[0].grid(alpha=0.2, axis='y')

    axes[1].bar(labels, acc_vals, color=['tab:blue', 'tab:orange'])
    axes[1].set_title('Validation Accuracy (higher better)')
    axes[1].set_ylim(0.0, 1.05)
    axes[1].grid(alpha=0.2, axis='y')

    axes[2].bar(labels, log_vals, color=['tab:blue', 'tab:orange'])
    axes[2].set_title('Validation Logloss (lower better)')
    axes[2].grid(alpha=0.2, axis='y')

    for ax, vals in zip(axes, [auc_vals, acc_vals, log_vals]):
        for i, v in enumerate(vals):
            if np.isfinite(v):
                ax.text(i, v, f"{float(v):.4f}", ha='center', va='bottom')

    plt.suptitle('Baseline vs Augmented (1 Epoch)')
    plt.tight_layout()

    out_path = Path(PATHS['plots_dir']) / 'baseline_vs_augmented_v51_e1.png'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=160, bbox_inches='tight')
    plt.show()
    print('Saved baseline-vs-aug plot to:', out_path)


In [ ]:

import sys
import json
import subprocess

EXPORTS = []
for run in RUNS:
    if run.get('status') != 'trained':
        print('Skipping export for run:', run['profile'], '| status=', run.get('status'))
        continue

    print('\nEXPORTING:', run.get('active_experiment_name', run['experiment_name']))
    output = subprocess.check_output([
        sys.executable,
        'dogs_vs_cats/src/export_experiment_artifacts.py',
        '--paths-config', PATHS_CFG,
        '--experiment-config', run['cfg_path'],
    ]).decode()
    info = json.loads(output)
    EXPORTS.append(info)
    print('Exported dir:', info['exported_dir'])
    print('Manifest:', info['manifest_path'])
    print('Bundle index:', info['bundle_index_path'])


In [ ]:

from pathlib import Path

for run in RUNS:
    print('\\n' + '=' * 90)
    print('PROFILE:', run['profile'])
    print('STATUS:', run.get('status'))
    print('EXPERIMENT:', run.get('active_experiment_name', run['experiment_name']))
    if run.get('error_message'):
        print('ERROR:', run['error_message'])

    summary = run.get('summary')
    if not summary:
        print('No summary available.')
        continue

    print('MONITOR:', summary.get('monitor', ''))
    print('BEST MONITOR:', summary.get('best_monitor'))
    print('BEST EPOCH:', summary.get('best_epoch'))
    files = summary.get('files', {})
    for key in sorted(files.keys()):
        p = Path(files[key])
        print('-', key, '->', p, '| exists=', p.exists())
